In [ ]:
#!/usr/bin/env python3
"""
gsMap Three-Tier Spatial Transcriptomics Mapping Pipeline
==========================================================
Tier 1: Human Visium tissue section (10x Genomics)
Tier 2: Human adult brain MERFISH (BA17 V1/V2 visual cortex)
Tier 3: Human fetal brain MERFISH (GW34, 7 cortical areas)

Each tier: data preparation -> gsMap steps 1-5 -> visualization
"""

import os
import sys
import time
import subprocess
import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path


# ========================= Configuration =========================

# --- GWAS parameters (fill in before running) ---
GWAS_FILE  = "./data/gwas/your_gwas_summary_stats.txt"   # Raw GWAS summary statistics file
TRAIT_NAME = "demo_trait"   # Trait identifier
GWAS_SEP   = "\t"
GWAS_COL_SNP           = "SNP"
GWAS_COL_EFFECT_ALLELE = "effect_allele"
GWAS_COL_OTHER_ALLELE  = "other_allele"
GWAS_COL_BETA          = "beta"
GWAS_COL_SE            = "se"
GWAS_COL_N             = "N"

# --- Base paths (fill in before running) ---
BASE_DIR       = "./results/"   # Project base directory
GSMAP_RESOURCE = "./data/gsmap_resource/"   # gsMap resource directory

# --- Spatial data paths (fill in before running) ---
VISIUM_H5      = "./data/visium/filtered_feature_bc_matrix.h5"   # Visium filtered feature barcode matrix h5
VISIUM_SPATIAL = "./data/visium/spatial/"   # Visium spatial directory
ADULT_MERFISH  = "./data/visium/adult_cortex.h5ad"   # Human adult cortex MERFISH h5ad
FETAL_GW34     = "./data/visium/fetal_gw34.h5ad"   # Human fetal GW34 MERFISH h5ad

# --- Compute resources ---
NUM_PARALLEL_CHROMS = 22
NUM_PROCESSES_LDSC  = 32

# --- Select which tiers to run ---
RUN_VISIUM        = True
RUN_ADULT_MERFISH = True
RUN_FETAL_GW34    = True


# ========================= Auto-configured =========================

GSMAP_BIN = os.path.join(os.path.dirname(sys.executable), "gsmap")
if not os.path.exists(GSMAP_BIN):
    GSMAP_BIN = "gsmap"


# ========================= Utilities =========================

def log(msg, level="INFO"):
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] {level} | {msg}")


def run_gsmap(args, step_name):
    """Run a gsmap CLI command and return success status."""
    log(f"Starting {step_name}...")
    cmd = [GSMAP_BIN] + args
    process = subprocess.run(cmd, capture_output=True, text=True)
    if process.returncode == 0:
        log(f"{step_name} completed")
        return True
    else:
        log(f"{step_name} failed", "ERROR")
        log(process.stderr[-1000:], "ERROR")
        return False


def _run_chrom_global(args):
    """Run LD score generation for a single chromosome."""
    gsmap_bin, workdir, sample_name, chrom, resource = args
    cmd = [gsmap_bin, "run_generate_ldscore",
           "--workdir", workdir, "--sample_name", sample_name,
           "--chrom", str(chrom),
           "--bfile_root", f"{resource}/LD_Reference_Panel/1000G_EUR_Phase3_plink/1000G.EUR.QC",
           "--keep_snp_root", f"{resource}/LDSC_resource/hapmap3_snps/hm",
           "--gtf_annotation_file", f"{resource}/genome_annotation/gtf/gencode.v46lift37.basic.annotation.gtf",
           "--gene_window_size", "50000"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return chrom, result.returncode


# ========================= Step 0: Format GWAS =========================

def format_gwas():
    """Format raw GWAS into gsMap-compatible sumstats.gz."""
    log("=" * 60)
    log("Step 0: Format GWAS summary statistics")
    log("=" * 60)

    output_path = os.path.join(BASE_DIR, f"{TRAIT_NAME}.sumstats.gz")
    if os.path.exists(output_path):
        log(f"GWAS already formatted, skipping: {output_path}")
        return output_path

    gwas = pd.read_csv(GWAS_FILE, sep=GWAS_SEP)
    log(f"Raw GWAS: {len(gwas)} SNPs")

    gwas["Z"] = gwas[GWAS_COL_BETA] / gwas[GWAS_COL_SE]
    gwas = gwas.dropna(subset=[GWAS_COL_SNP, GWAS_COL_EFFECT_ALLELE,
                                GWAS_COL_OTHER_ALLELE, "Z", GWAS_COL_N])
    gwas = gwas[np.isfinite(gwas["Z"])]
    gwas = gwas[gwas["Z"].abs() < 40]
    gwas = gwas[gwas[GWAS_COL_N] > 0]

    valid_alleles = {"A", "T", "C", "G"}
    gwas = gwas[gwas[GWAS_COL_EFFECT_ALLELE].str.upper().isin(valid_alleles)]
    gwas = gwas[gwas[GWAS_COL_OTHER_ALLELE].str.upper().isin(valid_alleles)]
    gwas = gwas.drop_duplicates(subset=[GWAS_COL_SNP], keep="first")

    gwas_fmt = pd.DataFrame({
        "SNP": gwas[GWAS_COL_SNP],
        "A1": gwas[GWAS_COL_EFFECT_ALLELE].str.upper(),
        "A2": gwas[GWAS_COL_OTHER_ALLELE].str.upper(),
        "Z": gwas["Z"],
        "N": gwas[GWAS_COL_N],
    })

    gwas_fmt.to_csv(output_path, sep="\t", index=False, compression="gzip")
    log(f"GWAS formatted: {output_path} ({len(gwas_fmt)} SNPs)")
    return output_path


# ========================= gsMap Core Pipeline =========================

def run_gsmap_pipeline(h5ad_path, gwas_path, sample_name, workdir):
    """Run gsMap steps 1-5 with checkpoint resumption."""
    os.makedirs(workdir, exist_ok=True)

    # --- Step 1: Find latent representations ---
    step1_done = os.path.exists(os.path.join(workdir, sample_name, "find_latent_representations"))
    if not step1_done:
        try:
            adata_check = sc.read_h5ad(h5ad_path, backed="r")
            if "latent_GVAE" in adata_check.obsm:
                step1_done = True
        except Exception:
            pass

    if step1_done:
        log("Step 1 already completed, skipping")
    else:
        if not run_gsmap(["run_find_latent_representations",
                          "--workdir", workdir, "--sample_name", sample_name,
                          "--input_hdf5_path", h5ad_path,
                          "--annotation", "annotation", "--data_layer", "count"],
                         "Step 1: find_latent_representations"):
            return False

    # --- Step 2: Latent to gene ---
    l2g_dir = os.path.join(workdir, sample_name, "latent_to_gene")
    if os.path.exists(l2g_dir) and len(os.listdir(l2g_dir)) > 0:
        feather_files = [f for f in os.listdir(l2g_dir) if f.endswith(".feather")]
        if feather_files and os.path.getsize(os.path.join(l2g_dir, feather_files[0])) > 1000:
            log("Step 2 already completed, skipping")
        else:
            log("Step 2 files corrupted, rerunning", "WARN")
            import shutil
            shutil.rmtree(l2g_dir)
            if not run_gsmap(["run_latent_to_gene",
                              "--workdir", workdir, "--sample_name", sample_name,
                              "--annotation", "annotation",
                              "--latent_representation", "latent_GVAE",
                              "--num_neighbour", "51", "--num_neighbour_spatial", "201"],
                             "Step 2: latent_to_gene"):
                return False
    else:
        if not run_gsmap(["run_latent_to_gene",
                          "--workdir", workdir, "--sample_name", sample_name,
                          "--annotation", "annotation",
                          "--latent_representation", "latent_GVAE",
                          "--num_neighbour", "51", "--num_neighbour_spatial", "201"],
                         "Step 2: latent_to_gene"):
            return False

    # --- Step 3: Generate LD scores (22 chromosomes in parallel) ---
    log("Step 3: generate_ldscore (22 chromosomes in parallel)")
    ldscore_dir = os.path.join(workdir, sample_name, "generate_ldscore")
    step3_done = False
    if os.path.exists(ldscore_dir):
        chunks = [f for f in os.listdir(ldscore_dir) if f.startswith(f"{sample_name}_chunk")]
        if chunks:
            sample_chunk = os.path.join(ldscore_dir, chunks[0])
            if os.path.exists(sample_chunk) and len(os.listdir(sample_chunk)) >= 66:
                step3_done = True
                log("Step 3 already completed, skipping")

    if not step3_done:
        from concurrent.futures import ProcessPoolExecutor, as_completed

        chrom_args = [(GSMAP_BIN, workdir, sample_name, c, GSMAP_RESOURCE) for c in range(1, 23)]
        with ProcessPoolExecutor(max_workers=NUM_PARALLEL_CHROMS) as executor:
            futures = {executor.submit(_run_chrom_global, args): args[3] for args in chrom_args}
            for future in as_completed(futures):
                chrom, rc = future.result()
                status = "OK" if rc == 0 else "FAIL"
                log(f"  {status} chr{chrom}")

        # Verify and rerun missing chromosomes
        if os.path.exists(ldscore_dir):
            chunks = [f for f in os.listdir(ldscore_dir) if f.startswith(f"{sample_name}_chunk")]
            if chunks:
                sample_chunk = os.path.join(ldscore_dir, chunks[0])
                if os.path.exists(sample_chunk):
                    n_files = len(os.listdir(sample_chunk))
                    if n_files < 66:
                        log(f"Incomplete ({n_files}/66), rerunning missing...", "WARN")
                        chroms_found = set()
                        for f in os.listdir(sample_chunk):
                            if ".l2.ldscore.feather" in f:
                                chroms_found.add(int(f.split(".")[1]))
                        missing = set(range(1, 23)) - chroms_found
                        if missing:
                            missing_args = [(GSMAP_BIN, workdir, sample_name, c, GSMAP_RESOURCE) for c in missing]
                            with ProcessPoolExecutor(max_workers=len(missing)) as executor:
                                futures = {executor.submit(_run_chrom_global, args): args[3] for args in missing_args}
                                for future in as_completed(futures):
                                    chrom, rc = future.result()
                                    status = "OK" if rc == 0 else "FAIL"
                                    log(f"  Rerun {status} chr{chrom}")

        log("Step 3 completed")

    # --- Step 4: Spatial LDSC ---
    ldsc_result = os.path.join(workdir, sample_name, "spatial_ldsc",
                               f"{sample_name}_{TRAIT_NAME}.csv.gz")
    if os.path.exists(ldsc_result):
        log("Step 4 already completed, skipping")
    else:
        if not run_gsmap(["run_spatial_ldsc",
                          "--workdir", workdir, "--sample_name", sample_name,
                          "--trait_name", TRAIT_NAME,
                          "--sumstats_file", gwas_path,
                          "--w_file", f"{GSMAP_RESOURCE}/LDSC_resource/weights_hm3_no_hla/weights.",
                          "--num_processes", str(NUM_PROCESSES_LDSC)],
                         "Step 4: spatial_ldsc"):
            return False

    # --- Step 5: Cauchy combination ---
    cauchy_result = os.path.join(workdir, sample_name, "cauchy_combination",
                                 f"{sample_name}_{TRAIT_NAME}.Cauchy.csv.gz")
    if os.path.exists(cauchy_result):
        log("Step 5 already completed, skipping")
    else:
        if not run_gsmap(["run_cauchy_combination",
                          "--workdir", workdir, "--sample_name", sample_name,
                          "--trait_name", TRAIT_NAME, "--annotation", "annotation"],
                         "Step 5: cauchy_combination"):
            return False

    return True


# ========================= Visualization =========================

def run_visualization(h5ad_path, workdir, sample_name, fig_dir, title_prefix,
                      region_col=None, has_depth=False):
    """Generate publication-quality figures from gsMap results."""
    import matplotlib
    matplotlib.rcParams["font.family"] = "DejaVu Sans"
    matplotlib.rcParams["pdf.fonttype"] = 42
    import matplotlib.pyplot as plt

    log("Generating figures...")
    os.makedirs(fig_dir, exist_ok=True)

    adata = sc.read_h5ad(h5ad_path)
    ldsc = pd.read_csv(f"{workdir}/{sample_name}/spatial_ldsc/{sample_name}_{TRAIT_NAME}.csv.gz")
    cauchy = pd.read_csv(f"{workdir}/{sample_name}/cauchy_combination/{sample_name}_{TRAIT_NAME}.Cauchy.csv.gz")

    # Align cell barcodes
    ldsc["spot"] = ldsc["spot"].astype(str)
    adata.obs_names = adata.obs_names.astype(str)
    ldsc = ldsc.set_index("spot")
    common = adata.obs_names.intersection(ldsc.index)
    log(f"  Matched cells: {len(common)}")

    if len(common) == 0:
        log("No matched cells, skipping visualization", "ERROR")
        return

    adata = adata[common, :]
    ldsc = ldsc.loc[common]

    adata.obs = adata.obs.copy()
    adata.obs["gsmap_p"] = ldsc["p"].values
    adata.obs["gsmap_neglog10p"] = -np.log10(ldsc["p"].values + 1e-300)
    clip_val = np.percentile(adata.obs["gsmap_neglog10p"], 99)
    adata.obs["gsmap_clip"] = np.clip(adata.obs["gsmap_neglog10p"], 0, clip_val)

    coords = adata.obsm["spatial"]
    cauchy_sorted = cauchy.sort_values("p_cauchy")

    log(f"\nCell type association results:\n{cauchy_sorted.to_string(index=False)}")

    # --- Figure 1: Three-panel spatial map ---
    fig, axes = plt.subplots(1, 3, figsize=(27, 8))

    # Panel 1: Cell types
    ax = axes[0]
    for ct in adata.obs["annotation"].unique():
        mask = (adata.obs["annotation"] == ct).values
        ax.scatter(coords[mask, 0], coords[mask, 1], s=0.5, alpha=0.5, label=ct, rasterized=True)
    ax.legend(fontsize=7, markerscale=10, loc="lower right")
    ax.set_title(f"{title_prefix}\nCell Types", fontsize=13, fontweight="bold")
    ax.set_aspect("equal")
    ax.axis("off")

    # Panel 2: Regions or subtypes
    ax = axes[1]
    if region_col and region_col in adata.obs.columns:
        for region in adata.obs[region_col].unique():
            mask = (adata.obs[region_col] == region).values
            ax.scatter(coords[mask, 0], coords[mask, 1], s=0.5, alpha=0.5, label=region, rasterized=True)
        ax.legend(fontsize=7, markerscale=10, loc="lower right")
        ax.set_title(f"{title_prefix}\nBrain Regions", fontsize=13, fontweight="bold")
    else:
        h2_col = [c for c in adata.obs.columns if "H2" in c]
        if h2_col:
            for sub in adata.obs[h2_col[0]].unique():
                mask = (adata.obs[h2_col[0]] == sub).values
                ax.scatter(coords[mask, 0], coords[mask, 1], s=0.5, alpha=0.5, rasterized=True)
        ax.set_title(f"{title_prefix}\nCell Subtypes", fontsize=13, fontweight="bold")
    ax.set_aspect("equal")
    ax.axis("off")

    # Panel 3: gsMap results
    ax = axes[2]
    ax.scatter(coords[:, 0], coords[:, 1], c="#EEEEEE", s=0.5, alpha=0.3, rasterized=True)
    sc_plot = ax.scatter(coords[:, 0], coords[:, 1], c=adata.obs["gsmap_clip"].values,
                         cmap="Reds", s=0.5, alpha=0.8, rasterized=True)
    cbar = plt.colorbar(sc_plot, ax=ax, shrink=0.7, pad=0.02)
    cbar.set_label("-log10(P value)", fontsize=11)
    ax.set_title(f"gsMap: {TRAIT_NAME}\nSpatial Association", fontsize=13, fontweight="bold")
    ax.set_aspect("equal")
    ax.axis("off")

    plt.suptitle(f"Genetically Informed Spatial Mapping of {TRAIT_NAME}\non {title_prefix}",
                 fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{fig_dir}/Fig_spatial_mapping.pdf", dpi=300, bbox_inches="tight")
    plt.savefig(f"{fig_dir}/Fig_spatial_mapping.png", dpi=300, bbox_inches="tight")
    plt.close()
    log("Figure 1: three-panel spatial map saved")

    # --- Figure 2: Cell type association bar chart ---
    fig, ax = plt.subplots(figsize=(10, 8))
    cauchy_sorted["neglog10p"] = -np.log10(cauchy_sorted["p_cauchy"] + 1e-300)
    n_types = len(cauchy_sorted)
    colors = ["#c0392b" if p < 0.05 / n_types else "#e74c3c" if p < 0.05 else "#bdc3c7"
              for p in cauchy_sorted["p_cauchy"]]
    ax.barh(range(n_types), cauchy_sorted["neglog10p"], color=colors, edgecolor="white")
    ax.set_yticks(range(n_types))
    ax.set_yticklabels(cauchy_sorted["annotation"], fontsize=9)
    ax.set_xlabel("-log10(P value)", fontsize=12)
    ax.set_title(f"{title_prefix}\nCell Type Association with {TRAIT_NAME}", fontsize=13, fontweight="bold")
    ax.axvline(x=-np.log10(0.05 / n_types), color="red", linestyle="--", linewidth=1, label="Bonferroni")
    ax.axvline(x=-np.log10(0.05), color="orange", linestyle="--", linewidth=1, label="Nominal")
    ax.legend(loc="lower right")
    ax.invert_yaxis()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(f"{fig_dir}/Fig_celltype_association.pdf", dpi=300, bbox_inches="tight")
    plt.savefig(f"{fig_dir}/Fig_celltype_association.png", dpi=300, bbox_inches="tight")
    plt.close()
    log("Figure 2: cell type bar chart saved")

    # --- Figure 3: Brain region association ---
    if region_col and region_col in adata.obs.columns:
        regions = adata.obs[region_col].dropna().unique()
        region_results = []
        for region in regions:
            if str(region) == "nan":
                continue
            mask = (adata.obs[region_col] == region).values
            pvals = adata.obs.loc[mask, "gsmap_p"].values
            weights = np.ones(len(pvals)) / len(pvals)
            cauchy_stats = np.sum(weights * np.tan((0.5 - pvals) * np.pi))
            cauchy_p = 0.5 - np.arctan(cauchy_stats) / np.pi
            region_results.append({"Brain Region": region, "p_cauchy": cauchy_p, "n_cells": mask.sum()})

        region_df = pd.DataFrame(region_results).sort_values("p_cauchy")
        region_df["neglog10p"] = -np.log10(region_df["p_cauchy"] + 1e-300)

        log(f"\nBrain region association results:\n{region_df.to_string(index=False)}")

        fig, ax = plt.subplots(figsize=(8, 5))
        n_r = len(region_df)
        colors_r = ["#c0392b" if p < 0.05 / n_r else "#e74c3c" if p < 0.05 else "#bdc3c7"
                     for p in region_df["p_cauchy"]]
        ax.barh(range(n_r), region_df["neglog10p"], color=colors_r, edgecolor="white")
        ax.set_yticks(range(n_r))
        ax.set_yticklabels(region_df["Brain Region"], fontsize=10)
        ax.set_xlabel("-log10(P value)", fontsize=12)
        ax.set_title(f"{title_prefix}\nBrain Region Association with {TRAIT_NAME}",
                     fontsize=13, fontweight="bold")
        ax.axvline(x=-np.log10(0.05 / n_r), color="red", linestyle="--", linewidth=1, label="Bonferroni")
        ax.legend(loc="lower right")
        ax.invert_yaxis()
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        plt.tight_layout()
        plt.savefig(f"{fig_dir}/Fig_region_association.pdf", dpi=300, bbox_inches="tight")
        plt.savefig(f"{fig_dir}/Fig_region_association.png", dpi=300, bbox_inches="tight")
        plt.close()
        region_df.to_csv(f"{fig_dir}/region_results.csv", index=False)
        log("Figure 3: brain region bar chart saved")

    # --- Figure 4: Cortical depth profile ---
    if has_depth and "cortical_depth" in adata.obs.columns:
        depth = adata.obs["cortical_depth"].values.astype(float)
        trait = adata.obs["gsmap_neglog10p"].values
        valid = ~np.isnan(depth)
        depth_v = depth[valid]
        trait_v = trait[valid]

        if len(depth_v) > 100:
            fig, ax = plt.subplots(figsize=(10, 6))
            n_bins = 50
            depth_bins = np.linspace(np.nanmin(depth_v), np.nanmax(depth_v), n_bins + 1)
            bin_means, bin_centers = [], []
            for j in range(n_bins):
                mask = (depth_v >= depth_bins[j]) & (depth_v < depth_bins[j + 1])
                if mask.sum() > 10:
                    bin_means.append(np.mean(trait_v[mask]))
                    bin_centers.append((depth_bins[j] + depth_bins[j + 1]) / 2)

            ax.plot(bin_centers, bin_means, color="#c0392b", linewidth=2)
            ax.fill_between(bin_centers, bin_means, alpha=0.2, color="#c0392b")
            ax.set_xlabel("Cortical Depth (superficial -> deep)", fontsize=12)
            ax.set_ylabel("Mean -log10(P value)", fontsize=12)
            ax.set_title(f"{TRAIT_NAME} Association Across Cortical Depth\n{title_prefix}",
                         fontsize=13, fontweight="bold")
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            plt.tight_layout()
            plt.savefig(f"{fig_dir}/Fig_cortical_depth.pdf", dpi=300, bbox_inches="tight")
            plt.savefig(f"{fig_dir}/Fig_cortical_depth.png", dpi=300, bbox_inches="tight")
            plt.close()
            log("Figure 4: cortical depth profile saved")

    log(f"All figures saved to: {fig_dir}")


# ========================= Tier 1: Human Visium =========================

def run_visium(gwas_path):
    """Tier 1: Human brain Visium 10x spatial mapping."""
    log("=" * 60)
    log("Tier 1: Human Visium tissue section mapping")
    log("=" * 60)

    workdir = os.path.join(BASE_DIR, f"results_visium_{TRAIT_NAME}")
    sample_name = f"human_visium_{TRAIT_NAME}"
    h5ad_path = os.path.join(BASE_DIR, f"human_visium_for_gsmap_{TRAIT_NAME}.h5ad")

    # Prepare data
    if not os.path.exists(h5ad_path):
        log("Preparing Visium data...")
        adata = sc.read_10x_h5(VISIUM_H5)

        import json
        with open(os.path.join(VISIUM_SPATIAL, "scalefactors_json.json")) as f:
            scale = json.load(f)
        positions = pd.read_csv(os.path.join(VISIUM_SPATIAL, "tissue_positions_list.csv"),
                                header=None, index_col=0)
        positions.columns = ["in_tissue", "row", "col", "pxl_row", "pxl_col"]
        positions = positions[positions["in_tissue"] == 1]

        common = adata.obs_names.intersection(positions.index)
        adata = adata[common, :]
        positions = positions.loc[common]
        adata.obsm["spatial"] = positions[["pxl_col", "pxl_row"]].values.astype(float)

        # Clustering for annotation
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=2000)
        sc.tl.pca(adata)
        sc.pp.neighbors(adata)
        sc.tl.leiden(adata, resolution=0.5)
        adata.obs["annotation"] = adata.obs["leiden"].copy()

        # Store raw counts
        adata_raw = sc.read_10x_h5(VISIUM_H5)
        adata_raw = adata_raw[common, :]
        adata.layers["count"] = adata_raw.X.copy()

        adata.var_names_make_unique()
        adata.write_h5ad(h5ad_path)
        log(f"Visium data prepared: {adata.n_obs} spots")
    else:
        log("Visium data already exists, skipping")

    # gsMap pipeline
    if run_gsmap_pipeline(h5ad_path, gwas_path, sample_name, workdir):
        fig_dir = os.path.join(BASE_DIR, f"figures_visium_{TRAIT_NAME}")
        run_visualization(h5ad_path, workdir, sample_name, fig_dir,
                          "Human Brain Visium (10x)", region_col=None, has_depth=False)
    else:
        log("Visium pipeline failed", "ERROR")


# ========================= Tier 2: Human Adult MERFISH =========================

def run_adult_merfish(gwas_path):
    """Tier 2: Human adult visual cortex MERFISH mapping."""
    log("=" * 60)
    log("Tier 2: Human adult brain MERFISH mapping")
    log("=" * 60)

    workdir = os.path.join(BASE_DIR, f"results_adult_merfish_{TRAIT_NAME}")
    sample_name = f"adult_merfish_{TRAIT_NAME}"
    h5ad_path = os.path.join(BASE_DIR, f"adult_merfish_for_gsmap_{TRAIT_NAME}.h5ad")

    # Prepare data
    if not os.path.exists(h5ad_path):
        log("Preparing adult MERFISH data...")
        adata = sc.read_h5ad(ADULT_MERFISH)
        log(f"Raw: {adata.n_obs} cells, {adata.n_vars} genes")

        adata.obs["annotation"] = adata.obs["H1_annotation"].copy()
        adata.obs_names_make_unique()
        adata.layers["count"] = adata.X.copy()
        adata.var_names = adata.var_names.astype(str)
        adata.var_names_make_unique()

        adata.write_h5ad(h5ad_path)
        log(f"Adult MERFISH data prepared: {adata.n_obs} cells")
    else:
        log("Adult MERFISH data already exists, skipping")

    # gsMap pipeline
    if run_gsmap_pipeline(h5ad_path, gwas_path, sample_name, workdir):
        fig_dir = os.path.join(BASE_DIR, f"figures_adult_merfish_{TRAIT_NAME}")
        run_visualization(h5ad_path, workdir, sample_name, fig_dir,
                          "Human Adult Visual Cortex (MERFISH)",
                          region_col="area", has_depth=True)
    else:
        log("Adult MERFISH pipeline failed", "ERROR")


# ========================= Tier 3: Human Fetal GW34 MERFISH =========================

def run_fetal_gw34(gwas_path):
    """Tier 3: Human fetal brain GW34 MERFISH mapping."""
    log("=" * 60)
    log("Tier 3: Human fetal brain GW34 MERFISH mapping")
    log("=" * 60)

    workdir = os.path.join(BASE_DIR, f"results_fetal_gw34_{TRAIT_NAME}")
    sample_name = f"fetal_gw34_{TRAIT_NAME}"
    h5ad_path = os.path.join(BASE_DIR, f"fetal_gw34_for_gsmap_{TRAIT_NAME}.h5ad")

    # Prepare data
    if not os.path.exists(h5ad_path):
        log("Preparing fetal GW34 data...")
        adata = sc.read_h5ad(FETAL_GW34)
        log(f"Raw: {adata.n_obs} cells, {adata.n_vars} genes")

        # Downsample per cell type
        np.random.seed(42)
        max_per_type = 5000
        sample_indices = []
        for ct in adata.obs["H2_annotation"].unique():
            idx = np.where(adata.obs["H2_annotation"].values == ct)[0]
            n_sample = min(len(idx), max_per_type)
            sampled = np.random.choice(idx, n_sample, replace=False)
            sample_indices.extend(sampled)
            log(f"  {ct}: {len(idx)} -> {n_sample}")

        sample_indices = sorted(sample_indices)
        adata_sub = adata[sample_indices, :].copy()

        adata_sub.obs_names_make_unique()
        adata_sub.obs["annotation"] = adata_sub.obs["H1_annotation"].copy()
        adata_sub.layers["count"] = adata_sub.X.copy()
        adata_sub.var_names = adata_sub.var_names.astype(str)
        adata_sub.var_names_make_unique()

        adata_sub.write_h5ad(h5ad_path)
        log(f"Fetal GW34 data prepared: {adata_sub.n_obs} cells, "
            f"{adata_sub.obs['region'].nunique()} regions")
    else:
        log("Fetal GW34 data already exists, skipping")

    # gsMap pipeline
    if run_gsmap_pipeline(h5ad_path, gwas_path, sample_name, workdir):
        fig_dir = os.path.join(BASE_DIR, f"figures_fetal_gw34_{TRAIT_NAME}")
        run_visualization(h5ad_path, workdir, sample_name, fig_dir,
                          "Human Fetal Cortex GW34 (MERFISH, 7 Areas)",
                          region_col="region", has_depth=False)
    else:
        log("Fetal GW34 pipeline failed", "ERROR")


# ========================= Main =========================

def main():
    log("=" * 60)
    log("gsMap Three-Tier Spatial Transcriptomics Pipeline")
    log(f"Trait: {TRAIT_NAME}")
    log(f"Tiers: Visium={RUN_VISIUM}, Adult MERFISH={RUN_ADULT_MERFISH}, Fetal GW34={RUN_FETAL_GW34}")
    log("=" * 60)

    # Format GWAS
    gwas_path = format_gwas()
    if not gwas_path:
        log("GWAS formatting failed", "ERROR")
        return

    # Tier 1: Visium
    if RUN_VISIUM:
        run_visium(gwas_path)

    # Tier 2: Adult MERFISH
    if RUN_ADULT_MERFISH:
        run_adult_merfish(gwas_path)

    # Tier 3: Fetal GW34
    if RUN_FETAL_GW34:
        run_fetal_gw34(gwas_path)

    log("=" * 60)
    log("Pipeline completed")
    log("=" * 60)


if __name__ == "__main__":
    main()